# Module 5 — Notebook 05b: Martin LIANA at fibroblast-subcluster resolution

Notebook 05 pooled all fibroblasts into one compartment, which dilutes IL11
(confined to subcluster 7) below LIANA's expression-proportion threshold.
This notebook re-runs LIANA with fibroblasts split by their notebook-04
subclusters — IAF (7), CHI3L1-high (6/15/3), other fibroblasts — and asks
only whether the IL11 -> IL11RA loop appears, and where.

Analyses of public datasets; all data credited to original authors.


In [1]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li

base = os.path.expanduser("~/Documents/immune-endotypes/data/martin_2019")
adata = sc.read_h5ad(os.path.join(base, "adata_martin_qc.h5ad"))
fib = sc.read_h5ad(os.path.join(base, "adata_martin_fibroblast.h5ad"))
COMP = {"0": "T", "1": "T", "8": "T", "9": "T", "2": "B", "3": "Plasma", "7": "Plasma",
        "12": "Plasma", "13": "Plasma", "4": "Myeloid", "5": "Epithelial", "6": "Epithelial",
        "10": "Fibroblast", "11": "Endothelial", "14": "Other"}
adata.obs["label"] = adata.obs["leiden_0.5"].astype(str).map(COMP)
fmap = {}
for cell, sub in zip(fib.obs_names, fib.obs["fib_leiden"].astype(str)):
    fmap[cell] = "Fib_IAF" if sub == "7" else ("Fib_CHI3L1" if sub in {"6", "15", "3"} else "Fib_other")
adata.obs["label"] = [fmap.get(c, l) for c, l in zip(adata.obs_names, adata.obs["label"])]
print(adata.obs.groupby(["label", "condition"]).size().unstack(fill_value=0))


condition    Involved  Uninvolved
label                            
B                9868        4073
Endothelial      1309        1573
Epithelial       3288        7840
Fib_CHI3L1        445           2
Fib_IAF           148           1
Fib_other         745        1601
Myeloid          3767        2668
Other             431         523
Plasma           8058        9296
T               28873       30520


In [2]:
rng = np.random.default_rng(0)
keep = []
for (cond, lab), idx in adata.obs.groupby(["condition", "label"]).groups.items():
    idx = np.asarray(idx)
    keep.extend(rng.choice(idx, size=min(1000, len(idx)), replace=False) if len(idx) > 1000 else idx)
sub = adata[keep].copy()
print("subsample:", sub.shape)
res = {}
for cond in ["Uninvolved", "Involved"]:
    a = sub[sub.obs["condition"] == cond].copy()
    li.mt.rank_aggregate(a, groupby="label", expr_prop=0.1,
                         resource_name="consensus", n_perms=100, use_raw=False,
                         key_added=f"liana_{cond}", verbose=False)
    r = a.uns[f"liana_{cond}"].copy()
    r["condition"] = cond
    res[cond] = r
    r.to_csv(f"../results/liana_martin_fib_{cond}.csv", index=False)
    print(cond, r.shape)


subsample: (15295, 25356)
Uninvolved (2765, 14)
Involved (8349, 14)


In [3]:
# The IL11 question, all source/target rows
for cond, r in res.items():
    m = r[(r["ligand_complex"] == "IL11") & (r["receptor_complex"].str.contains("IL11RA", regex=False))]
    print(f"--- {cond}: IL11-IL11RA rows:", len(m))
    if len(m):
        print(m.nsmallest(5, "magnitude_rank")[["source", "target", "magnitude_rank"]].assign(strength=lambda d: (1 / d["magnitude_rank"]).round(2)).to_string(index=False))
# And re-check the OSM switch at this resolution
for cond, r in res.items():
    m = r[(r["ligand_complex"] == "OSM") & (r["receptor_complex"].str.contains("OSMR", regex=False))]
    if len(m):
        b = m.loc[m["magnitude_rank"].idxmin()]
        print(cond, "OSM-OSMR best:", b["source"], "->", b["target"], round(1 / b["magnitude_rank"], 2))
    else:
        print(cond, "OSM-OSMR: absent")


--- Uninvolved: IL11-IL11RA rows: 0
--- Involved: IL11-IL11RA rows: 0
Uninvolved OSM-OSMR: absent
Involved OSM-OSMR best: Myeloid -> Fib_CHI3L1 2.39


## Read-out (fill in after running)

- IL11-IL11RA: present at subcluster resolution? Involved-only, source/target
  Fib_IAF (autocrine) or Fib_IAF -> other fibroblasts?
- OSM-OSMR at this resolution: does the switch hold, and is the target now
  specifically Fib_IAF / Fib_CHI3L1?
- Next: notebook 06 — human integration; notebook 07 — Kinchen mouse arm;
  then the cross-cohort summary figure.
